<a href="https://colab.research.google.com/github/SP50-cell/Bcom-lectures/blob/main/Slides4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==============================================================================
# CELL 1: SETUP, ENVIRONMENT CONFIGURATION, AND GOOGLE DRIVE MOUNT
# ==============================================================================
# Instructions: Run this first. It installs dependencies, mounts Google Drive,
# and initializes the Gemini API client safely.

!pip install -q google-genai gTTS pydub
import os
from google.colab import drive
from google import genai

# Mounting Google Drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# Retrieve API key securely from Colab Secrets, with a direct input fallback if needed
api_key = None
try:
    from google.colab import userdata
    api_key = userdata.get('Gemini_A')
except Exception:
    pass

if not api_key:
    # Fallback: if secret lookup fails, prompt safely or use environment variable
    api_key = os.environ.get("GEMINI_API_KEY", "")

if not api_key:
    print("⚠️ WARNING: 'Gemini_A' secret not found via userdata. Please paste your API key below:")
    import getpass
    api_key = getpass.getpass("Enter your Gemini API Key: ")

client = genai.Client(api_key=api_key)

print("Environment setup complete! Drive mounted and API client initialized successfully.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 1.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
huggingface-hub 1.28.0 requires click<9.0.0,>=8.4.2, but you have click 8.1.8 which is incompatible.
spacy 3.8.16 requires click<9.0.0,>=8.2.1, but you have click 8.1.8 which is incompatible.
wandb 0.28.1 requires click>=8.2.0, but you have click 8.1.8 which is incompatible.
Mounted at /content/drive
⚠️ WARNING: 'Gemini_A' secret not found via userdata. Please paste your API key below:
Enter your Gemini API Key: ··········
Environment setup complete! Drive mounted and API client initialized successfully.


In [2]:
# ==============================================================================
# CELL 2: CORE PIPELINE, MODEL SELECTION, AND UPDATED HTML GENERATOR
# ==============================================================================

import os
import time
from google.genai import types

# ------------------------------------------------------------------------------
# CONFIGURATION & ANNOTATION MAPPING
# ------------------------------------------------------------------------------
SUBJECT_CODE = "BOS"  # Options: AB, BOS, BOP, GE, ET
UNIT_NUM = "1"
LECTURE_NUM = "1"

SUBJECT_MAP = {
    "AB": "Fundamentals of Accounting",
    "BOS": "Business Organisation",
    "BOP": "Principles of Business Management",
    "GE": "General English",
    "ET": "Economic Theory"
}

subject_name = SUBJECT_MAP.get(SUBJECT_CODE, "Business Organisation")

DRIVE_BASE = "/content/drive/MyDrive/NotebookLM"
SUBFOLDER_PATH = os.path.join(DRIVE_BASE, SUBJECT_CODE)
TEXTBOOK_PATH = os.path.join("/content/drive/MyDrive/TextbookLM", f"{SUBJECT_CODE} Textbook.pdf")

mp4_filename = f"{SUBJECT_CODE} Unit {UNIT_NUM} Lecture {LECTURE_NUM}.mp4"
mp4_path = os.path.join(SUBFOLDER_PATH, mp4_filename)

output_html_name = f"{SUBJECT_CODE} Unit {UNIT_NUM} Lecture {LECTURE_NUM}.html"
output_md_name = f"{SUBJECT_CODE} Unit {UNIT_NUM} Lecture {LECTURE_NUM}.md"

output_html_path = os.path.join(SUBFOLDER_PATH, output_html_name)
output_md_path = os.path.join(SUBFOLDER_PATH, output_md_name)

os.makedirs(SUBFOLDER_PATH, exist_ok=True)
os.makedirs("/content/drive/MyDrive/TextbookLM", exist_ok=True)

# ------------------------------------------------------------------------------
# MODEL SELECTION
# ------------------------------------------------------------------------------
PRIMARY_MODEL = "gemini-3.8-flash"
BACKUP_MODEL = "gemini-3.5-flash-lite"

# ------------------------------------------------------------------------------
# CONTENT EXTRACTION & AI PROCESSING
# ------------------------------------------------------------------------------
print(f"[{time.strftime('%X')}] Processing {subject_name} - Unit {UNIT_NUM}, Lecture {LECTURE_NUM}...")
start_time = time.time()
uploaded_bytes_total = 0

video_file = None
pdf_file = None

if os.path.exists(mp4_path):
    print(f"Uploading video: {mp4_filename}")
    uploaded_bytes_total += os.path.getsize(mp4_path)
    video_file = client.files.upload(file=mp4_path)

if os.path.exists(TEXTBOOK_PATH):
    print(f"Uploading textbook: {SUBJECT_CODE} Textbook.pdf")
    uploaded_bytes_total += os.path.getsize(TEXTBOOK_PATH)
    pdf_file = client.files.upload(file=TEXTBOOK_PATH)

prompt_payload = f"""
You are acting as a Senior Professor & Head of Commerce Department (B.Com Lead Mentor) for Semester 1 students.
Analyze the provided video lecture and Unit {UNIT_NUM} of the textbook ({SUBJECT_CODE} Textbook.pdf).

CRITICAL FORMATTING & CONTENT RULES:
1. Output strictly in clean semantic HTML tags (<h1>, <h2>, <h3>, <p>, <ul>, <li>, <strong>, <em>). DO NOT use Markdown symbols (#, **, ```).
2. DO NOT create or use any tables whatsoever.
3. Index / Table of Contents items in the main document must have tight spacing.
4. Detailed Lecture Notes must be concise and shortened by half (keep it brief, high-yield, and focused).
5. Audio Transcript MUST be formatted as ONE single giant continuous paragraph. Strip out ALL timestamps (e.g., [00:00 - 00:27]).
6. Slide Transcript MUST be formatted as ONE single giant continuous paragraph. Strip out ALL slide numbers, headers, or markers.
7. Textbook vs Lecture Difference Analysis: Treat the textbook unit as the core baseline. If the textbook contains material that the lecture omits, consider that normal (the textbook is the core). Only flag a difference if the lecture introduces unique core concepts, definitions, or principles that are entirely missing from the textbook. Do NOT count examples, case studies, or omissions as differences. If no unique concepts exist in the lecture outside the textbook, write a short statement saying no unique differences found. Otherwise, write a very brief paragraph highlighting only those specific conceptual differences.

Generate structured HTML content containing:
- Document Header Title: {subject_name} - Unit {UNIT_NUM} Lecture {LECTURE_NUM}
- Index / Table of Contents items listed tightly in sequence.
- Sub-topics List of Content.
- Detailed Lecture Notes (shortened by half).
- Single-paragraph Audio Transcript.
- Single-paragraph Slide Transcript.
- Brief Textbook vs Lecture Difference Comparison Paragraph.
"""

contents = [prompt_payload]
if video_file: contents.append(video_file)
if pdf_file: contents.append(pdf_file)

used_model = PRIMARY_MODEL
try:
    response = client.models.generate_content(model=PRIMARY_MODEL, contents=contents)
except Exception as e:
    print(f"Primary model ({PRIMARY_MODEL}) failed, switching to backup model {BACKUP_MODEL}. Error: {e}")
    used_model = BACKUP_MODEL
    response = client.models.generate_content(model=BACKUP_MODEL, contents=contents)

ai_output_text = response.text
execution_time = round(time.time() - start_time, 2)

usage_metadata = getattr(response, 'usage_metadata', None)
input_tokens = getattr(usage_metadata, 'prompt_token_count', None) if usage_metadata else None

if input_tokens is None:
    try:
        token_count_resp = client.models.count_tokens(model=used_model, contents=contents)
        input_tokens = token_count_resp.total_tokens
    except Exception:
        input_tokens = "N/A"

output_tokens = getattr(usage_metadata, 'candidates_token_count', 0) if usage_metadata else "N/A"
response_bytes = len(ai_output_text.encode('utf-8'))
total_internet_mb = round((uploaded_bytes_total + response_bytes) / (1024 * 1024), 2)

cleaned_html = ai_output_text.replace("```html", "").replace("```", "")

# ------------------------------------------------------------------------------
# UPDATED HTML TEMPLATE WITH GOOGLE DOCS LIGHT THEME TOGGLE ON 'B' BUTTON
# ------------------------------------------------------------------------------
html_content = f"""<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>{subject_name} - Unit {UNIT_NUM} Lecture {LECTURE_NUM}</title>
    <style>
        :root {{
            --doc-bg: #1e1e1e;
            --toolbar-bg: #2d2d2d;
            --menu-bg: #222222;
            --text-main: #e0e0e0;
            --border-color: #444444;
            --hover-bg: #3c4043;
            --page-bg: #121212;
            --transcript-bg: #161616;
            --docs-icon-bg: #555555;
            --tts-btn-bg: #444444;
            --tts-btn-hover: #555555;
            --stop-btn-bg: #222222;
            --stop-btn-hover: #333333;
            --sidebar-bg: #222222;
            --heading-color: #ffffff;
        }}

        body {{
            background-color: var(--page-bg);
            color: var(--text-main);
            font-family: Arial, sans-serif;
            margin: 0;
            padding: 0;
            transition: background-color 0.2s ease, color 0.2s ease;
        }}

        /* Google Docs Light Theme Class (Toggled via Bold 'B' button) */
        body.light-theme {{
            --doc-bg: #ffffff;
            --toolbar-bg: #f1f3f4;
            --menu-bg: #f8f9fa;
            --text-main: #202124;
            --border-color: #dadce0;
            --hover-bg: #e8eaed;
            --page-bg: #f8f9fa;
            --transcript-bg: #f1f3f4;
            --docs-icon-bg: #1a73e8;
            --tts-btn-bg: #1a73e8;
            --tts-btn-hover: #1557b0;
            --stop-btn-bg: #f1f3f4;
            --stop-btn-hover: #e8eaed;
            --sidebar-bg: #f8f9fa;
            --heading-color: #202124;
        }}

        /* Header Bar */
        .gdocs-header {{
            background-color: var(--menu-bg);
            border-bottom: 1px solid var(--border-color);
            padding: 8px 16px;
            display: flex;
            align-items: center;
            justify-content: space-between;
        }}
        .header-left {{
            display: flex;
            align-items: center;
            gap: 12px;
        }}
        .docs-icon {{
            background-color: var(--docs-icon-bg);
            color: white;
            font-weight: bold;
            padding: 6px 10px;
            border-radius: 4px;
            font-size: 14px;
        }}
        .doc-title-input {{
            background: transparent;
            border: 1px solid transparent;
            color: var(--text-main);
            font-size: 16px;
            padding: 4px 8px;
            border-radius: 4px;
            outline: none;
            width: 350px;
        }}
        .doc-title-input:hover, .doc-title-input:focus {{
            border-color: var(--border-color);
            background-color: var(--toolbar-bg);
        }}

        /* Menu Bar */
        .gdocs-menubar {{
            background-color: var(--menu-bg);
            display: flex;
            padding: 2px 16px;
            gap: 4px;
            font-size: 13px;
            border-bottom: 1px solid var(--border-color);
        }}
        .menu-item {{
            color: var(--text-main);
            padding: 4px 8px;
            border-radius: 4px;
            cursor: pointer;
        }}
        .menu-item:hover {{ background-color: var(--hover-bg); }}

        /* Toolbar */
        .gdocs-toolbar {{
            background-color: var(--toolbar-bg);
            border-bottom: 1px solid var(--border-color);
            padding: 6px 16px;
            display: flex;
            align-items: center;
            gap: 6px;
            flex-wrap: wrap;
        }}
        .tool-group {{
            display: flex;
            align-items: center;
            gap: 2px;
            border-right: 1px solid var(--border-color);
            padding-right: 8px;
            margin-right: 4px;
        }}
        .tool-btn {{
            background: transparent;
            border: 1px solid transparent;
            color: var(--text-main);
            padding: 5px 8px;
            border-radius: 4px;
            cursor: pointer;
            font-size: 14px;
        }}
        .tool-btn:hover {{ background-color: var(--hover-bg); border-color: var(--border-color); }}
        .select-box {{
            background-color: var(--doc-bg);
            color: var(--text-main);
            border: 1px solid var(--border-color);
            padding: 4px 6px;
            border-radius: 4px;
            font-size: 13px;
        }}
        .tts-action-btn {{
            background-color: var(--tts-btn-bg);
            color: white;
            border: none;
            padding: 6px 12px;
            border-radius: 4px;
            cursor: pointer;
            font-weight: bold;
            font-size: 13px;
        }}
        .tts-action-btn:hover {{ background-color: var(--tts-btn-hover); }}
        .stop-action-btn {{
            background-color: var(--stop-btn-bg);
            color: var(--text-main);
            border: 1px solid var(--border-color);
            padding: 6px 12px;
            border-radius: 4px;
            cursor: pointer;
            font-weight: bold;
            font-size: 13px;
        }}
        .stop-action-btn:hover {{ background-color: var(--stop-btn-hover); }}

        /* Layout with Left-Side Index Sidebar */
        .main-layout {{
            display: flex;
            max-width: 1200px;
            margin: 20px auto;
            gap: 20px;
        }}
        .left-index-sidebar {{
            width: 260px;
            background-color: var(--sidebar-bg);
            border: 1px solid var(--border-color);
            padding: 20px;
            border-radius: 4px;
            position: sticky;
            top: 20px;
            height: fit-content;
            font-size: 13px;
            box-sizing: border-box;
        }}
        .left-index-sidebar h3 {{
            color: var(--heading-color);
            font-weight: bold;
            margin-top: 0;
            font-size: 14px;
            border-bottom: 1px solid var(--border-color);
            padding-bottom: 8px;
        }}
        .left-index-sidebar ul {{
            padding-left: 15px;
            margin: 0;
        }}
        .left-index-sidebar li {{
            margin-bottom: 8px;
            line-height: 1.4;
        }}

        /* Document Paper Sheet Container */
        .doc-page-container {{
            flex-grow: 1;
            max-width: 816px;
            background-color: var(--doc-bg);
            border: 1px solid var(--border-color);
            padding: 72px;
            box-shadow: 0 4px 15px rgba(0,0,0,0.3);
            min-height: 1056px;
            box-sizing: border-box;
        }}

        /* Bold Headers */
        h1, h2, h3, h4, h5, h6 {{
            color: var(--heading-color) !important;
            font-weight: bold !important;
        }}
        p, li {{ line-height: 1.6; }}

        /* Reduced Spacing for Main Body Index / Ordered Lists */
        .doc-page-container ol, .doc-page-container ul {{
            margin-top: 4px;
            margin-bottom: 8px;
        }}
        .doc-page-container li {{
            margin-bottom: 2px !important;
            line-height: 1.2 !important;
        }}

        /* Audio & Slide Transcripts */
        .transcript-block {{
            font-size: 0.8em !important;
            color: var(--text-main);
            background-color: var(--transcript-bg);
            padding: 15px;
            border-left: 2px solid #555555;
            margin: 15px 0;
        }}
    </style>
</head>
<body>

    <!-- Header Bar -->
    <div class="gdocs-header">
        <div class="header-left">
            <div class="docs-icon">Doc</div>
            <input type="text" class="doc-title-input" value="{subject_name} - Unit {UNIT_NUM} Lecture {LECTURE_NUM}">
        </div>
    </div>

    <!-- Menu Bar -->
    <div class="gdocs-menubar">
        <span class="menu-item">File</span>
        <span class="menu-item">Edit</span>
        <span class="menu-item">View</span>
        <span class="menu-item">Insert</span>
        <span class="menu-item">Format</span>
        <span class="menu-item">Tools</span>
        <span class="menu-item">Extensions</span>
        <span class="menu-item">Help</span>
    </div>

    <!-- Toolbar -->
    <div class="gdocs-toolbar">
        <div class="tool-group">
            <button class="tool-btn" title="Undo">↩</button>
            <button class="tool-btn" title="Redo">↪</button>
            <button class="tool-btn" title="Print">🖨️</button>
        </div>
        <div class="tool-group">
            <select class="select-box">
                <option>Normal text</option>
                <option>Heading 1</option>
                <option>Heading 2</option>
            </select>
            <select class="select-box" id="globalTextSize" onchange="changeGlobalTextSize(this.value)">
                <option value="12px">11</option>
                <option value="14px">14</option>
                <option value="16px" selected>16</option>
                <option value="18px">18</option>
            </select>
        </div>
        <div class="tool-group">
            <button class="tool-btn" style="font-weight:bold;" onclick="toggleDocsTheme()" title="Toggle Google Docs Light/Dark Theme">B</button>
            <button class="tool-btn" style="font-style:italic;">I</button>
            <button class="tool-btn" style="text-decoration:underline;">U</button>
        </div>
        <div class="tool-group" style="border:none;">
            <button class="tts-action-btn" onclick="startReadAloud()">🔊 Read Out Loud</button>
            <button class="stop-action-btn" onclick="stopReadAloud()">⏹ Stop</button>
        </div>
    </div>

    <!-- Main Layout with Left Index Sidebar and Document Sheet -->
    <div class="main-layout">
        <!-- Left Side Index Sidebar -->
        <div class="left-index-sidebar">
            <h3>Index & Topics</h3>
            <ul>
                <li>Introduction to core concepts and foundational definitions.</li>
                <li>Detailed lecture breakdown and structured notes.</li>
                <li>Comprehensive unified audio lecture transcript.</li>
                <li>Unified presentation slide transcript summary.</li>
                <li>Textbook vs Lecture content divergence analysis.</li>
            </ul>
        </div>

        <!-- Paper Sheet Container -->
        <div class="doc-page-container" id="documentContent">
            {cleaned_html}
        </div>
    </div>

    <script>
        function changeGlobalTextSize(size) {{
            document.getElementById('documentContent').style.fontSize = size;
        }}

        function toggleDocsTheme() {{
            document.body.classList.toggle('light-theme');
        }}

        let synth = window.speechSynthesis;
        let utterance = null;

        function startReadAloud() {{
            if (synth.speaking) {{ synth.cancel(); }}
            const textToRead = document.getElementById('documentContent').innerText;
            utterance = new SpeechSynthesisUtterance(textToRead);
            utterance.rate = 1.0;
            synth.speak(utterance);
        }}

        function stopReadAloud() {{
            if (synth.speaking) {{ synth.cancel(); }}
        }}
    </script>
</body>
</html>
"""

# ------------------------------------------------------------------------------
# FILE WRITING & METRICS REPORT
# ------------------------------------------------------------------------------
for path in [output_html_path, output_md_path]:
    if os.path.exists(path):
        os.remove(path)

with open(output_html_path, "w", encoding="utf-8") as f:
    f.write(html_content)

with open(output_md_path, "w", encoding="utf-8") as f:
    f.write(ai_output_text)

print("=======================================================")
print("📊 PROCESSING METRICS REPORT")
print("=======================================================")
print(f"⏱️ Execution Time    : {execution_time} seconds")
print(f"🤖 Model Used        : {used_model}")
print(f"📥 Input API Tokens  : {input_tokens}")
print(f"📤 Output API Tokens : {output_tokens}")
print(f"🌐 Internet Data Used: {total_internet_mb} MB")
print(f"📁 Generated HTML    : {output_html_path}")
print(f"📁 Generated MD      : {output_md_path}")
print("=======================================================")

[17:07:51] Processing Business Organisation - Unit 1, Lecture 1...
Uploading video: BOS Unit 1 Lecture 1.mp4
Uploading textbook: BOS Textbook.pdf
📊 PROCESSING METRICS REPORT
⏱️ Execution Time    : 124.89 seconds
🤖 Model Used        : gemini-3.8-flash
📥 Input API Tokens  : 399556
📤 Output API Tokens : 6691
🌐 Internet Data Used: 422.65 MB
📁 Generated HTML    : /content/drive/MyDrive/NotebookLM/BOS/BOS Unit 1 Lecture 1.html
📁 Generated MD      : /content/drive/MyDrive/NotebookLM/BOS/BOS Unit 1 Lecture 1.md
